# Evaluating trained models

First, to evaluate a trained model we need to load it from a checkpoint file.  We can use this using the `load_inference_model_from_checkpoint` helper function.  We will consider a model that exists in the testing directory.

In [1]:
from modelforge.utils.io import get_path_string
from modelforge.tests import data

ckpt_file = get_path_string(data) + "/best_SchNet-qm9-epoch=433.ckpt"

In [2]:
from modelforge.potential.potential import load_inference_model_from_checkpoint

potential = load_inference_model_from_checkpoint(ckpt_file, use_training_mode_neighborlist=True)

2026-10-19 23:54:20.269 | DEBUG    | modelforge.potential.potential:generate_potential:928 - training_parameter=None
2026-10-19 23:54:20.270 | DEBUG    | modelforge.potential.potential:generate_potential:929 - potential_parameter=SchNetParameters(potential_name='SchNet', only_unique_pairs=False, core_parameter=CoreParameter(number_of_radial_basis_functions=51, maximum_interaction_radius=0.49999999999999994, number_of_interaction_modules=3, number_of_filters=64, shared_interactions=False, activation_function_parameter=ActivationFunctionConfig(activation_function_name='ShiftedSoftplus', activation_function_arguments=None, activation_function=ShiftedSoftplus()), featurization=Featurization(properties_to_featurize=['atomic_number'], atomic_number=AtomicNumber(maximum_atomic_number=101, number_of_per_atom_features=64), atomic_period=AtomicPeriod(maximum_period=8, number_of_per_period_features=32), atomic_group=AtomicGroup(maximum_group=18, number_of_per_group_features=32)), predicted_proper

Note, we set `use_training_mode_neighborlist=True` in this example, as the training mode neighborlist is designed to work with batches, whereas the inference mode neighborlist is not.

## Loading a dataset
To evaluate the different subsets (i.e., training, validation, and test) of a given dataset, we will create an instance of the `DataModule` class.  To correctly break up the dataset into the various subsets, the `DataModule` will need to know what scheme was used for splitting the dataset (e.g., `RandomRecordSplittingStrategy`) as well as the seed used for data splitting.  Additionally, any other arguments, such as the filtering by element, we also need to specify. 

To make this easier, there is a helper function `initialize_datamodule_from_config` that will read in the full config .toml file used to train the model, and return the instance of the `DataModule`.  Note this function can optionalyl accept the `local_cache_dir`, `dataset_cache_dir`, and `local_yaml_file`, in cases where the model is being evaluated on a different machine than where it was trained, with a different file structure.  Specifying values for these arguments will overwrite what is in the .toml file. 

In [3]:
from modelforge.dataset.dataset import initialize_datamodule_from_config

config_file = get_path_string(data) + "/config.toml"

dm = initialize_datamodule_from_config(config_file)

2026-10-19 23:54:20.289 | INFO     | modelforge.train.training:read_config:2278 - Reading config from : /home/cri/PycharmProjects/modelforge/modelforge/tests/data/config.toml
2026-10-19 23:54:20.290 | DEBUG    | modelforge.dataset.dataset:__init__:351 - Loading config data from /home/cri/PycharmProjects/modelforge/modelforge/dataset/yaml_files/qm9.yaml
2026-10-19 23:54:20.295 | INFO     | modelforge.dataset.dataset:__init__:391 - Using dataset version nc_1000_v1.1
2026-10-19 23:54:20.295 | DEBUG    | modelforge.dataset.dataset:_acquire_dataset:449 - Unzipped npz file qm9.npz already exists in ./cache
2026-10-19 23:54:20.295 | DEBUG    | modelforge.dataset.dataset:_file_validation:747 - Validating file ./cache/qm9.npz
2026-10-19 23:54:20.296 | DEBUG    | modelforge.dataset.dataset:_from_file_cache:1133 - Loading processed data from ./cache/qm9.npz generated on 2026-10-19 23:35:34.735179
2026-10-19 23:54:20.296 | DEBUG    | modelforge.dataset.dataset:_from_file_cache:1136 - Properties of

We can examine the instance of the `DataModule` in various ways. 

First, we can access the entire dataset if we choose; e.g., let us grab the first index, which returns a instance of `BatchData`, that contains an instance of `NNPInput` (i.e., the input we would give the potential) and `Metadata` (which contains information used as reference for training). 

In [4]:
dm.torch_dataset[0]

BatchData(nnp_input=<modelforge.utils.prop.NNPInput object at 0x72e05cd05cb0>, metadata=<modelforge.utils.prop.Metadata object at 0x72e05cd06c70>)

In [5]:
dm.torch_dataset.number_of_records

1000

We can easily pass this directly to the `Potential` instance:

In [6]:
potential(dm.torch_dataset[0].nnp_input)

{'per_atom_scalar_representation': tensor([[ 1.4055e-02, -4.3460e+00,  3.6395e+00, -1.8440e+00, -4.7277e-01,
          -1.8369e+00,  1.3188e-01, -1.4007e+00, -1.2535e+00,  1.6851e+00,
           1.3097e+00, -8.5496e-01,  2.0650e+00,  1.1775e+00, -1.3773e+00,
          -1.0531e+00, -8.2265e-01,  1.7301e+00, -1.1399e+00, -2.2953e-02,
           1.1919e+00, -3.2259e+00, -9.6208e-01,  2.9343e+00,  1.7135e+00,
           1.0048e-01,  3.5138e-01, -1.2392e+00, -1.4657e+00, -1.8975e+00,
          -1.0716e+00, -2.4881e+00, -9.8163e+00, -1.4606e+00, -1.5558e+00,
           1.3516e+00,  2.2103e+00,  7.8273e-01,  7.9594e-01, -1.8992e+00,
          -4.5259e-01,  1.7921e+00, -3.5329e+00,  3.7835e-01,  1.9299e+00,
          -5.4592e-01,  6.1587e-01, -1.0419e-01,  9.9306e-01,  3.8341e+00,
          -2.9925e-01, -2.3630e+00, -2.6463e+00, -2.0083e-01,  9.4083e-01,
           1.9011e+00, -8.2728e-01, -1.3348e+00,  6.1457e-01,  3.3860e+00,
          -1.8021e+00, -1.1042e-01,  1.1591e+00, -7.9951e-01],
   

When accessing `dm.torch_dataset` we do not necessarily know if this was in the training, validation, or test split.  

We can access these splits directly, calling `train_dataset`, `val_dataset` or `test_dataset`:

In [7]:
dm.test_dataset[0]

BatchData(nnp_input=<modelforge.utils.prop.NNPInput object at 0x72e05cd05d90>, metadata=<modelforge.utils.prop.Metadata object at 0x72e05cd06810>)

To be able to relate to the original indices in `dm.torch_dataset`, we can access the `indices` property, where below e can see that our first entry (0), corresponds to system 256 in the original dataset (since we used the `RandomRecordSplittingStrategy` which randomizes the order of the entries). 

In [8]:
dm.test_dataset.indices

[256,
 526,
 70,
 461,
 454,
 634,
 393,
 794,
 520,
 156,
 966,
 675,
 598,
 462,
 13,
 90,
 875,
 55,
 45,
 643,
 630,
 753,
 572,
 19,
 61,
 104,
 581,
 602,
 754,
 665,
 105,
 435,
 662,
 559,
 14,
 863,
 539,
 519,
 761,
 232,
 554,
 441,
 574,
 747,
 653,
 877,
 224,
 293,
 556,
 358,
 898,
 806,
 712,
 254,
 97,
 710,
 736,
 621,
 146,
 465,
 431,
 330,
 395,
 129,
 741,
 579,
 115,
 111,
 549,
 80,
 222,
 617,
 969,
 809,
 527,
 776,
 151,
 822,
 870,
 240,
 277,
 7,
 918,
 466,
 400,
 557,
 566,
 846,
 985,
 481,
 219,
 318,
 894,
 2,
 770,
 335,
 284,
 638,
 96,
 455]

Rather than evaluating each individual entry we can perform batch processing using the helper function `evaluate_batch`.  This function returns a dictionary of the output of evaluating the potential, along with any relevant metadata.  Additionally, each entry will also include the name of the record (as specified in the hdf5 file), allowing for easier examination. 

The function argument `subset` can accept either 'test', 'train', or 'val'. 

In [9]:
from modelforge.potential.potential import evaluate_batch

output = evaluate_batch(dm, potential, subset='test')

Note, the dictionary that is returned is keyed by the index (see above).  So, for example, the first configuration we evaluated would be 256.

In [10]:
output[256]

{'name': 'dsgdb9nsd_000257',
 'index': 256,
 'output': {'per_system_total_charge': tensor([0.]),
  'per_system_energy': tensor([-5469.7188], grad_fn=<SelectBackward0>),
  'per_atom_scalar_representation': tensor([[ 3.5721e-01, -5.6950e-01,  8.9329e-01, -8.6741e-01, -5.5377e-01,
            7.9049e-02,  1.8460e-01, -3.0355e-01, -5.3787e-01,  3.1008e-01,
           -1.4134e-01, -7.8582e-01, -5.9055e-01,  6.0473e-01, -6.8033e-01,
           -7.6756e-01, -1.0879e+00, -1.2051e-02,  7.1606e-02,  9.0758e-02,
            3.9859e-01, -1.9582e-01, -4.2990e-01,  1.1162e+00,  2.9817e-01,
           -1.7037e-01, -4.8819e-01, -3.7774e-01, -1.0145e+00,  8.7541e-02,
           -2.9997e-01, -1.5234e+00, -4.9042e+00,  6.2014e-02, -4.8165e-01,
           -2.9408e-01,  6.4694e-01,  8.4078e-01, -1.3456e-01, -1.9798e+00,
            1.2003e-01,  4.0471e-01, -1.1133e+00,  6.3509e-02,  8.5310e-03,
            1.9528e-01,  8.2183e-02, -3.3373e-01,  8.5537e-02,  1.1208e+00,
           -2.7947e-01, -6.5824e-01, 